# Outliers en Preprocesamiento de Datos

## Objetivo
En este cuaderno aprenderemos:

- Qué es un outlier o dato atípico.
- Qué impactos puede tener en el análisis de datos y en machine learning.
- Cómo detectarlo usando métodos estadísticos y visuales.
- Cómo tratarlo durante el preprocesamiento.

## Idea central
Un **outlier** es un valor que se aleja considerablemente del comportamiento general de los datos.  
Puede aparecer por:

- errores de medición,
- errores de digitación,
- corrupción de datos,
- o porque realmente corresponde a un caso extremo verdadero.

No siempre se debe eliminar un outlier. Antes de tomar una decisión, se debe analizar su causa y su efecto en el problema.

In [ ]:
# ==========================================
# CELDA 3: IMPORTAR LIBRERÍAS
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import LocalOutlierFactor
from scipy.stats.mstats import winsorize

# Configuración opcional para mejorar visualización
plt.rcParams["figure.figsize"] = (8, 5)

print("Librerías importadas correctamente.")

In [ ]:
# ==========================================
# CELDA 4: CREAR DATOS DE EJEMPLO
# ==========================================
# Generaremos un conjunto de datos sintético para mostrar
# cómo se comportan los outliers.

np.random.seed(42)

# Datos "normales"
edad = np.random.normal(loc=35, scale=8, size=100).round()
ingreso = np.random.normal(loc=700000, scale=120000, size=100).round()

# Agregamos algunos outliers manualmente
edad_outliers = np.array([5, 90, 95])
ingreso_outliers = np.array([2000000, 2500000, 3000000])

# Unimos los datos
edad_total = np.concatenate([edad, edad_outliers])
ingreso_total = np.concatenate([ingreso, ingreso_outliers])

# Creamos DataFrame
df = pd.DataFrame({
    "Edad": edad_total,
    "Ingreso": ingreso_total
})

# Mostramos primeras filas
df.head()

In [ ]:
# ==========================================
# CELDA 5: EXPLORACIÓN INICIAL
# ==========================================

print("Dimensiones del dataset:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

print("\nEstadísticas descriptivas:")
display(df.describe())

## Observación inicial

En este ejemplo trabajaremos con dos variables numéricas:

- **Edad**
- **Ingreso**

La mayor parte de los datos sigue un comportamiento relativamente normal, pero agregamos algunos valores extremos de forma intencional.  
Esto permitirá ver cómo cambian las estadísticas y cómo detectar outliers con distintos métodos.

In [ ]:
# ==========================================
# CELDA 7: BOXPLOTS
# ==========================================
# El boxplot permite detectar valores extremos
# fuera de los "bigotes".

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot(df["Edad"], vert=True)
axes[0].set_title("Boxplot de Edad")
axes[0].set_ylabel("Edad")

axes[1].boxplot(df["Ingreso"], vert=True)
axes[1].set_title("Boxplot de Ingreso")
axes[1].set_ylabel("Ingreso")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELDA 8: SCATTER PLOT
# ==========================================
# Útil para detectar puntos alejados en dos dimensiones.

plt.figure(figsize=(8, 5))
plt.scatter(df["Edad"], df["Ingreso"])
plt.title("Gráfico de dispersión: Edad vs Ingreso")
plt.xlabel("Edad")
plt.ylabel("Ingreso")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# CELDA 9: DETECCIÓN DE OUTLIERS CON IQR
# ==========================================
# Regla:
# Outlier si x < Q1 - 1.5*IQR o x > Q3 + 1.5*IQR

def detectar_outliers_iqr(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    mascara = (serie < limite_inferior) | (serie > limite_superior)

    return mascara, limite_inferior, limite_superior

# Aplicamos a Edad
mask_edad_iqr, li_edad, ls_edad = detectar_outliers_iqr(df["Edad"])

# Aplicamos a Ingreso
mask_ingreso_iqr, li_ingreso, ls_ingreso = detectar_outliers_iqr(df["Ingreso"])

print("Límites IQR para Edad:")
print("Inferior:", li_edad, "| Superior:", ls_edad)

print("\nLímites IQR para Ingreso:")
print("Inferior:", li_ingreso, "| Superior:", ls_ingreso)

print("\nCantidad de outliers en Edad (IQR):", mask_edad_iqr.sum())
print("Cantidad de outliers en Ingreso (IQR):", mask_ingreso_iqr.sum())

In [ ]:
# ==========================================
# CELDA 10: MOSTRAR FILAS CON OUTLIERS SEGÚN IQR
# ==========================================

outliers_iqr = df[mask_edad_iqr | mask_ingreso_iqr]

print("Filas detectadas como outliers por IQR:")
display(outliers_iqr)

In [ ]:
# ==========================================
# CELDA 11: DETECCIÓN CON REGLA DE 3 SIGMAS
# ==========================================
# Outlier si x < media - 3*desv_std o x > media + 3*desv_std

def detectar_outliers_3sigmas(serie):
    media = serie.mean()
    std = serie.std()

    limite_inferior = media - 3 * std
    limite_superior = media + 3 * std

    mascara = (serie < limite_inferior) | (serie > limite_superior)

    return mascara, limite_inferior, limite_superior

mask_edad_sigma, li_edad_sigma, ls_edad_sigma = detectar_outliers_3sigmas(df["Edad"])
mask_ingreso_sigma, li_ingreso_sigma, ls_ingreso_sigma = detectar_outliers_3sigmas(df["Ingreso"])

print("Límites 3 sigmas para Edad:")
print("Inferior:", round(li_edad_sigma, 2), "| Superior:", round(ls_edad_sigma, 2))

print("\nLímites 3 sigmas para Ingreso:")
print("Inferior:", round(li_ingreso_sigma, 2), "| Superior:", round(ls_ingreso_sigma, 2))

print("\nCantidad de outliers en Edad (3 sigmas):", mask_edad_sigma.sum())
print("Cantidad de outliers en Ingreso (3 sigmas):", mask_ingreso_sigma.sum())

In [ ]:
# ==========================================
# CELDA 12: DETECCIÓN CON LOCAL OUTLIER FACTOR (LOF)
# ==========================================
# LOF evalúa qué tan aislado está un punto respecto a sus vecinos.

X = df[["Edad", "Ingreso"]]

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
predicciones_lof = lof.fit_predict(X)

# En LOF:
# -1 = outlier
#  1 = normal
df["LOF_Outlier"] = predicciones_lof

print("Cantidad de outliers detectados por LOF:", (df["LOF_Outlier"] == -1).sum())

display(df[df["LOF_Outlier"] == -1])

In [ ]:
# ==========================================
# CELDA 13: VISUALIZACIÓN DE LOF
# ==========================================

normales = df[df["LOF_Outlier"] == 1]
atipicos = df[df["LOF_Outlier"] == -1]

plt.figure(figsize=(8, 5))
plt.scatter(normales["Edad"], normales["Ingreso"], label="Normales")
plt.scatter(atipicos["Edad"], atipicos["Ingreso"], label="Outliers")
plt.title("Outliers detectados con LOF")
plt.xlabel("Edad")
plt.ylabel("Ingreso")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# CELDA 14: IMPACTO DE LOS OUTLIERS EN LAS ESTADÍSTICAS
# ==========================================

# Tomaremos como criterio IQR para excluir outliers en Ingreso
df_sin_outliers = df[~(mask_edad_iqr | mask_ingreso_iqr)].copy()

comparacion = pd.DataFrame({
    "Media original": df[["Edad", "Ingreso"]].mean(),
    "Media sin outliers": df_sin_outliers[["Edad", "Ingreso"]].mean(),
    "Mediana original": df[["Edad", "Ingreso"]].median(),
    "Mediana sin outliers": df_sin_outliers[["Edad", "Ingreso"]].median(),
    "Desv. estándar original": df[["Edad", "Ingreso"]].std(),
    "Desv. estándar sin outliers": df_sin_outliers[["Edad", "Ingreso"]].std()
})

display(comparacion)

In [ ]:
# ==========================================
# CELDA 15: TRATAMIENTO 1 - ELIMINAR OUTLIERS
# ==========================================
# Se eliminan las filas detectadas como atípicas.

df_eliminado = df[~(mask_edad_iqr | mask_ingreso_iqr)].copy()

print("Cantidad de filas original:", len(df))
print("Cantidad de filas luego de eliminar outliers:", len(df_eliminado))

display(df_eliminado.head())

In [ ]:
# ==========================================
# CELDA 16: TRATAMIENTO 2 - TRUNCAR
# ==========================================
# Truncar significa quedarse solo con una parte central
# de la distribución y eliminar extremos según percentiles.

p_inf = 0.05
p_sup = 0.95

lim_inf_ingreso = df["Ingreso"].quantile(p_inf)
lim_sup_ingreso = df["Ingreso"].quantile(p_sup)

df_truncado = df[(df["Ingreso"] >= lim_inf_ingreso) & (df["Ingreso"] <= lim_sup_ingreso)].copy()

print("Límite inferior truncado:", lim_inf_ingreso)
print("Límite superior truncado:", lim_sup_ingreso)
print("Filas originales:", len(df))
print("Filas después de truncar:", len(df_truncado))

In [ ]:
# ==========================================
# CELDA 17: TRATAMIENTO 3 - WINSORIZAR
# ==========================================
# Winsorizar no elimina los valores extremos,
# sino que los reemplaza por límites percentilares.

df_wins = df.copy()

# Winsorización al 5% por cada lado en Ingreso
df_wins["Ingreso_wins"] = winsorize(df["Ingreso"], limits=[0.05, 0.05])

# Winsorización al 5% por cada lado en Edad
df_wins["Edad_wins"] = winsorize(df["Edad"], limits=[0.05, 0.05])

display(df_wins[["Edad", "Edad_wins", "Ingreso", "Ingreso_wins"]].head(10))

In [ ]:
# ==========================================
# CELDA 18: COMPARACIÓN VISUAL
# ==========================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot(df["Ingreso"])
axes[0].set_title("Ingreso original")

axes[1].boxplot(df_wins["Ingreso_wins"])
axes[1].set_title("Ingreso winsorizado")

plt.tight_layout()
plt.show()